In [ ]:
from comment import CommentRepository
from repository import Repository
from db_config import SessionLocal
from dotenv import load_dotenv
from util import to_text_without_leading_common_whitespace
import os

load_dotenv()
session = SessionLocal()
commentDao = CommentRepository()
repositoryDao = Repository()
comments = commentDao.get_comments_having_null_code(limit=100)

for comment in comments:
    repositoryEntity = repositoryDao.get_repository(comment.repository_id)
    file_location = os.getenv(
        'REPOSITORY_DIRECTORY') + '/' + comment.repository_directory + '/' + comment.file

    start_line_index = comment.start_line - 1
    end_line_index = comment.end_line - 1
    with open(file, 'r') as file:
        lines = file.readlines()
        lines_before = lines[max(0, start_line_index - 10):start_line_index]
        comment.code_before = to_text_without_leading_common_whitespace(lines_before)

        lines_after_begin_index = start_line_index if comment.text in lines[start_line_index] else end_line_index + 1
        lines_after = lines[lines_after_begin_index: min(lines_after_begin_index + 10, len(lines))]
        comment.code_after = to_text_without_leading_common_whitespace(lines_after)

    url = f'{repositoryEntity.repo_url}/blob/{repositoryEntity.commit_hash}/{comment.file}/#L{comment.start_line}'
    print(f'\n\n\n\n########################## {comment.id} #########################')

    print(
        f'Repository: {comment.repository_directory}\nFile:\n{file_location}:{comment.start_line}\nURL: {url}\nComment:\n\n{comment.text}\n')

    session.merge(comment)
    session.commit()

